# Results Analysis: Scaled-Down Deep Hedging Model

This notebook evaluates the trained "scaled-down" model using Yahoo Finance data. It generates NAV curves and compares performance against standard baselines.

In [ ]:
from pathlib import Path
import json, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import torch

sns.set_style("whitegrid")

# Setup paths
NB = Path.cwd()
ROOT = NB.parent if NB.name == 'notebooks' else NB
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

RUN_DIR = ROOT / "models/scaled_down_v2"
CKPT = RUN_DIR / "best.pt"
CONF = RUN_DIR / "config.json"
DATA_PATH = ROOT / "data/processed/cleaned/yahoo_v3_10feats.csv"

print(f"Using Run Directory: {RUN_DIR}")
assert CKPT.exists(), f"Missing checkpoint: {CKPT}. Please train the model first!"
assert CONF.exists(), f"Missing config: {CONF}"
assert DATA_PATH.exists(), f"Missing data: {DATA_PATH}"

In [ ]:
from rl_agent.train_ac_gae import PolicyValueNet, make_obs_fixer
from rl_agent.experiment import build_envs, deterministic_rewards

# 1. Load Data
panel = pd.read_csv(DATA_PATH, parse_dates=['date']).sort_values('date').reset_index(drop=True)
print(f"Panel range: {panel['date'].min()} to {panel['date'].max()}")

# 2. Load Config & Build Envs
cfg = json.loads(CONF.read_text())
envs = build_envs(
    panel=panel,
    features=cfg['features'],
    train_end=cfg['train_end'],
    valid_end=cfg['valid_end'],
    window=cfg['window'],
    txn_cost_bps=cfg['txn_cost_bps'],
    pos_limit=cfg['pos_limit'],
    rebalance_every=cfg.get('rebalance_every', 1),
    slippage_bps=cfg['slippage_bps']
)
env_tr, env_va, env_te = envs['env_tr'], envs['env_va'], envs['env_te']

# 3. Load Policy
to_fixed = make_obs_fixer(cfg['window'], len(cfg['features']))
net = PolicyValueNet(input_dim=env_tr.reset().size, hidden=cfg['hidden'])
net.load_state_dict(torch.load(CKPT, map_location='cpu'))
net.obs_fix = to_fixed
net.pos_limit = float(cfg['pos_limit'])

## 1. NAV Curves (Deterministic Rollout)
We compare the GAE policy's performance across Train, Valid, and Test splits.

In [ ]:
from rl_agent.metrics import nav_curve

plt.figure(figsize=(10, 5))
r_tr, nav_tr = nav_curve(env_tr, net, "Train")
r_va, nav_va = nav_curve(env_va, net, "Valid")
r_te, nav_te = nav_curve(env_te, net, "Test")

plt.axhline(1.0, color="grey", lw=0.8, ls="--")
plt.title("NAV Curves (Scaled-Down Model)")
plt.ylabel("NAV")
plt.legend()
plt.show()

## 2. Comparison vs Baselines
We compare against a simple "Long SPY" hold strategy and VIX-based overlays.

In [ ]:
from simulator.baselines import hold_policy, vix_band_policy, vix_vol_target
from rl_agent.deploy import load_policy
from collections import OrderedDict

def _as_actor(fn):
    if hasattr(fn, "act"): return fn
    class _Tmp: 
        def __init__(self, f): self.f = f
        def act(self, obs, deterministic=True): return float(self.f(obs)), None, None
    return _Tmp(fn)

# VIX stats for overlays
train_mask = panel['date'] <= pd.Timestamp(cfg['train_end'])
vix_mean = panel.loc[train_mask, 'vix'].mean()
vix_std = panel.loc[train_mask, 'vix'].std()
vix_idx = cfg['features'].index('vix')

policies = OrderedDict({
    "GAE Policy": net,
    "Long SPY": _as_actor(hold_policy(cfg['pos_limit'])),
    "VIX Band": _as_actor(vix_band_policy(vix_idx, 20.0, 30.0, 0.0, cfg['pos_limit'], vix_mean, vix_std)),
    "VIX Vol Target": _as_actor(vix_vol_target(vix_idx, 20.0, cfg['pos_limit'], vix_mean, vix_std))
})

results = []
for name, actor in policies.items():
    # Standard evaluation logic
    rew = deterministic_rewards(env_te, actor)
    sharpe = (rew.mean() / rew.std() * np.sqrt(252)) if rew.std() > 0 else 0
    results.append({"Policy": name, "Test Sharpe": sharpe, "Mean Bps": rew.mean()})

summary_df = pd.DataFrame(results)
print("Comparing Test Set Performance:")
display(summary_df)